# FeH-global MCMC convergence diagnostics

Use this notebook after copying the server sampling output to the local machine. It checks:

- chain traces and rank plots;
- split $\hat R$;
- bulk/tail effective sample sizes (ESS);
- Monte Carlo standard errors (MCSE);
- divergences, BFMI, acceptance probability, and tree depth when sampler diagnostics were saved;
- whether `outlier_u0` is accumulating at its lower bound of 30.

> The grouped `.npz` format below is strongly preferred. A flattened `.txt` can be reshaped only when the exact chain count and draws per chain are known, and it cannot recover divergences that were never saved.

## Recommended server-side export

The real-data runner now performs this export automatically when `--save-full-mcmc` is enabled (the default). The essential export format is:

```python
posterior = fitter.sampler.get_samples(group_by_chain=True)
sample_stats = fitter.sampler.get_extra_fields(group_by_chain=True)

payload = {}
for name, values in posterior.items():
    payload[f"posterior__{name}"] = np.asarray(values)
for name, values in sample_stats.items():
    payload[f"sample_stats__{name}"] = np.asarray(values)

np.savez_compressed(
    os.path.join(output_dir, "mcmc_grouped_with_diagnostics.npz"),
    **payload,
)
```

The runner explicitly collects `diverging`, `energy`, `potential_energy`, `num_steps`, and `accept_prob`, and derives `tree_depth` from `num_steps`.

In [ ]:
from pathlib import Path
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

az.style.use("arviz-whitegrid")
print(f"ArviZ {az.__version__}")

## Configuration

Set `MODEL_STAGE` to `core`, `quadratic`, or `cross`. The notebook selects the corresponding grouped result automatically. Keep `N_CHAINS` and `DRAWS_PER_CHAIN` only for legacy TXT files.

In [ ]:
RESULT_DIR = Path("/Users/ytwang/Library/CloudStorage/OneDrive-Personal/Files/postgraduate/PyProjects/Dyn/bayesian-binary-masses/results/data_diffpoly2d_anchor_cute_tfeherr_good")
MODEL_STAGE = "core"  # core, quadratic, or cross

# Preferred chain-aware file produced by run_on_data_feh_global.py.
grouped_candidates = sorted(RESULT_DIR.glob(f"mcmc_grouped_with_diagnostics_*stage-{MODEL_STAGE}_*.npz"))
if len(grouped_candidates) > 1:
    raise ValueError(f"Multiple grouped files match stage {MODEL_STAGE!r}: {grouped_candidates}")
GROUPED_NPZ = grouped_candidates[0] if grouped_candidates else RESULT_DIR / "missing_grouped.npz"

# Fallback for old runs that saved only flattened samples.
legacy_candidates = sorted(RESULT_DIR.glob(f"mcmc__*stage-{MODEL_STAGE}_*.txt"))
if len(legacy_candidates) > 1:
    raise ValueError(f"Multiple legacy files match stage {MODEL_STAGE!r}: {legacy_candidates}")
LEGACY_TXT = legacy_candidates[0] if legacy_candidates else None

N_CHAINS = 4
DRAWS_PER_CHAIN = 2000

# Column order written by the current DifferencePolyFehMLR fitter for this setup.
LEGACY_PARAM_NAMES = [
    "a0", "b1", "a1", "b2", "a2", "c_xy",
    "f_outlier", "outlier_u0", "outlier_sigma",
]
COEFF_NAMES = ["a0", "b1", "a1", "b2", "a2", "c_xy"]

print("Result directory:", RESULT_DIR)
print("Selected grouped file:", GROUPED_NPZ if GROUPED_NPZ.exists() else "none")
print("Selected legacy file:", LEGACY_TXT or "none")

In [ ]:
def _expand_coeffs(posterior, coeff_names):
    """Expand posterior['coeffs'][chain, draw, coefficient] into named variables."""
    if "coeffs" not in posterior:
        return posterior
    coeffs = np.asarray(posterior.pop("coeffs"))
    if coeffs.ndim != 3:
        raise ValueError(f"Expected coeffs with shape (chain, draw, coefficient), got {coeffs.shape}.")
    if coeffs.shape[-1] != len(coeff_names):
        raise ValueError(
            f"Coefficient width {coeffs.shape[-1]} does not match COEFF_NAMES ({len(coeff_names)})."
        )
    for index, name in enumerate(coeff_names):
        # Prefer explicitly sampled scalar sites if both representations exist.
        posterior.setdefault(name, coeffs[..., index])
    return posterior


def load_grouped_npz(path, coeff_names):
    with np.load(path, allow_pickle=False) as saved:
        posterior = {
            key.removeprefix("posterior__"): np.asarray(saved[key])
            for key in saved.files
            if key.startswith("posterior__")
        }
        sample_stats = {
            key.removeprefix("sample_stats__"): np.asarray(saved[key])
            for key in saved.files
            if key.startswith("sample_stats__")
        }
    if not posterior:
        raise ValueError("No posterior__* arrays were found in the grouped NPZ.")
    posterior = _expand_coeffs(posterior, coeff_names)
    return az.from_dict(posterior=posterior, sample_stats=sample_stats or None), "grouped NPZ"


def load_legacy_txt(path, param_names, n_chains, draws_per_chain):
    flat = np.atleast_2d(np.loadtxt(path))
    expected_rows = n_chains * draws_per_chain
    if flat.shape[0] != expected_rows:
        raise ValueError(
            f"TXT has {flat.shape[0]} rows, but N_CHAINS * DRAWS_PER_CHAIN = {expected_rows}. "
            "Do not guess these values: use the exact server settings."
        )
    if flat.shape[1] != len(param_names):
        raise ValueError(
            f"TXT has {flat.shape[1]} columns, but LEGACY_PARAM_NAMES has {len(param_names)} names."
        )
    chained = flat.reshape(n_chains, draws_per_chain, flat.shape[1])
    posterior = {name: chained[..., index] for index, name in enumerate(param_names)}
    warnings.warn(
        "Loaded a flattened TXT by assuming NumPyro's chain-major flattening order. "
        "Divergences and other sampler diagnostics are unavailable.",
        stacklevel=2,
    )
    return az.from_dict(posterior=posterior), "legacy flattened TXT"


if GROUPED_NPZ.exists():
    idata, source_format = load_grouped_npz(GROUPED_NPZ, COEFF_NAMES)
elif LEGACY_TXT is not None and Path(LEGACY_TXT).exists():
    idata, source_format = load_legacy_txt(
        Path(LEGACY_TXT), LEGACY_PARAM_NAMES, N_CHAINS, DRAWS_PER_CHAIN
    )
else:
    raise FileNotFoundError(
        "No result file found. Set GROUPED_NPZ or LEGACY_TXT in the configuration cell."
    )

actual_chains = idata.posterior.sizes["chain"]
actual_draws = idata.posterior.sizes["draw"]
print(f"Loaded {source_format}: {actual_chains} chains x {actual_draws} draws")
if actual_chains < 4:
    warnings.warn("Fewer than four chains were loaded; convergence assessment is weaker.", stacklevel=1)
idata

## Numerical diagnostics

Constant masked coefficients are reported separately and excluded from $\hat R$/ESS checks because these diagnostics are undefined for a zero-variance variable.

In [ ]:
constant_vars = []
diagnostic_vars = []
for name, values in idata.posterior.data_vars.items():
    array = np.asarray(values)
    if np.all(np.nanstd(array, axis=(0, 1)) == 0):
        constant_vars.append(name)
    else:
        diagnostic_vars.append(name)

print("Variables used for diagnostics:", diagnostic_vars)
print("Constant/masked variables excluded:", constant_vars or "none")

summary = az.summary(
    idata,
    var_names=diagnostic_vars,
    kind="all",
    stat_focus="mean",
    round_to=4,
)
display(summary)

In [ ]:
R_HAT_LIMIT = 1.01
MIN_ESS = 400

failed = pd.DataFrame(index=summary.index)
failed["r_hat"] = summary.get("r_hat", np.nan)
failed["ess_bulk"] = summary.get("ess_bulk", np.nan)
failed["ess_tail"] = summary.get("ess_tail", np.nan)
failed["bad_r_hat"] = failed["r_hat"] >= R_HAT_LIMIT
failed["low_bulk_ess"] = failed["ess_bulk"] < MIN_ESS
failed["low_tail_ess"] = failed["ess_tail"] < MIN_ESS
problem_rows = failed[failed[["bad_r_hat", "low_bulk_ess", "low_tail_ess"]].any(axis=1)]

if problem_rows.empty:
    print(f"PASS: all non-constant parameters have R-hat < {R_HAT_LIMIT} and bulk/tail ESS >= {MIN_ESS}.")
else:
    print("Parameters requiring attention:")
    display(problem_rows)

## Trace and rank plots

The chains should overlap without long-term drift. Rank histograms should be approximately uniform rather than showing chain-specific shapes.

In [ ]:
preferred_plot_vars = [
    "a0", "b1", "a1", "b2", "a2", "c_xy",
    "f_outlier", "outlier_u0", "outlier_sigma",
]
plot_vars = [name for name in preferred_plot_vars if name in diagnostic_vars]
if not plot_vars:
    plot_vars = diagnostic_vars[:6]

az.plot_trace(idata, var_names=plot_vars, compact=False, figsize=(12, 2.4 * len(plot_vars)))
plt.tight_layout()
plt.show()

az.plot_rank(idata, var_names=plot_vars, kind="bars", figsize=(12, 2.4 * len(plot_vars)))
plt.tight_layout()
plt.show()

## Divergences and energy diagnostics

In [ ]:
if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:
    divergent = np.asarray(idata.sample_stats["diverging"], dtype=bool)
    per_chain = divergent.sum(axis=1)
    print("Divergences per chain:", per_chain.tolist())
    print(f"Total divergences: {divergent.sum()} / {divergent.size}")
    if divergent.any():
        print("FAIL: divergences should be zero. Inspect the affected posterior regions before trusting the fit.")
    else:
        print("PASS: no divergences.")
else:
    print("Divergence information is unavailable; use the grouped server export for the next run.")

if hasattr(idata, "sample_stats") and "energy" in idata.sample_stats:
    bfmi = np.asarray(az.bfmi(idata))
    print("BFMI per chain:", np.round(bfmi, 3).tolist())
    if np.any(bfmi < 0.3):
        print("WARNING: BFMI below 0.3 indicates poor exploration of the energy distribution.")
else:
    print("Energy was not saved, so BFMI is unavailable.")

if hasattr(idata, "sample_stats") and "accept_prob" in idata.sample_stats:
    accept_prob = np.asarray(idata.sample_stats["accept_prob"])
    print("Mean acceptance probability per chain:", np.round(accept_prob.mean(axis=1), 3).tolist())

MAX_TREE_DEPTH = 10  # NumPyro NUTS default unless overridden in the runner
if hasattr(idata, "sample_stats") and "tree_depth" in idata.sample_stats:
    tree_depth = np.asarray(idata.sample_stats["tree_depth"])
    saturated = tree_depth >= MAX_TREE_DEPTH
    print("Maximum tree depth per chain:", tree_depth.max(axis=1).tolist())
    print(f"Transitions at max tree depth: {saturated.sum()} / {saturated.size}")
    if saturated.any():
        print("WARNING: some transitions saturated max_tree_depth.")
else:
    print("Tree-depth information is unavailable.")

## Outlier-boundary check

In [ ]:
OUTLIER_U0_LOWER_BOUND = 30.0
BOUNDARY_TOLERANCE = 0.5

if "outlier_u0" in idata.posterior:
    values = np.asarray(idata.posterior["outlier_u0"])
    near_bound = values < OUTLIER_U0_LOWER_BOUND + BOUNDARY_TOLERANCE
    fraction = near_bound.mean()
    print(f"Fraction of outlier_u0 samples below {OUTLIER_U0_LOWER_BOUND + BOUNDARY_TOLERANCE:.1f}: {fraction:.3%}")
    print("outlier_u0 percentiles:", np.round(np.percentile(values, [1, 5, 16, 50, 84, 95, 99]), 4))
    if fraction > 0.05:
        print("WARNING: appreciable posterior mass lies near the hard lower bound.")
else:
    print("outlier_u0 is not present in this posterior.")

## Interpretation checklist

Treat the run as numerically acceptable only when:

1. all scientifically relevant parameters have $\hat R<1.01$;
2. bulk and tail ESS are both at least 400 (larger is preferable);
3. trace plots have no drift or chain-specific modes;
4. rank plots are approximately uniform;
5. there are zero divergences;
6. `outlier_u0` is not artificially pinned to 30;
7. repeated runs with different seeds give compatible posterior summaries.

Passing these checks establishes numerical convergence, not physical correctness of the likelihood or data model.